In [1]:
# =============================================================================
# Autoencoders & Variational Autoencoders
# =============================================================================
# ---------------------------------------------------------------------------
# SCENE 1 — you describe a face to a sketch artist
# ---------------------------------------------------------------------------
# You see a person. You cannot send the artist a photo. You can only say
# a few facts:
#   "round face, glasses, short hair, smiling"
#
# The artist draws from those facts. The drawing is close, not pixel-perfect.
#
# That's an autoencoder:
#   YOU          = encoder   (turn a rich thing into a short description)
#   THE FACTS    = z         (the short description — also called "latent")
#   THE ARTIST   = decoder   (turn the short description back into a picture)
#
# Nobody cares if the sketch is 100% identical. We care that the SHORT
# DESCRIPTION captured what mattered.
#
#
# ---------------------------------------------------------------------------
# SCENE 2 — zip files (almost the same, but dumber)
# ---------------------------------------------------------------------------
# Zip:  big file → small file → unzip back to the same bytes.
# Zip is a FIXED recipe. It does not learn.
#
# An autoencoder LEARNS its own recipe from examples:
#   see many photos  →  get good at "what to keep in the short description"
#   so unzip-the-neural-way still looks like the original photo.
#
# Zip must be perfect. Autoencoders are allowed to be a bit lossy —
# like JPEG. A slightly blurry rebuild is OK if the gist survived.
#
#
# ---------------------------------------------------------------------------
# SCENE 3 — a tiny number example (so it's not only metaphors)
# ---------------------------------------------------------------------------
# Suppose each "photo" is just 4 numbers (imagine brightness of 4 pixels):
#
#   input x = [1.0,  0.9,  0.1,  0.0]
#
# Encoder squashes that to 2 numbers:
#   z = [0.8, -0.2]
#
# Decoder expands z back to 4 numbers:
#   rebuild x̂ = [0.95, 0.88, 0.12, 0.04]
#
# Close enough. Training's only job: nudge the encoder/decoder so this
# gap gets smaller on lots of examples.
#
# If we forced z to be 4 numbers too, the net could copy [1, 0.9, 0.1, 0]
# and "succeed" without understanding anything. That's why z is SMALLER
# than x — we make copying impossible. That's the bottleneck.
#
#
# ---------------------------------------------------------------------------
# THREE everyday uses (why anyone bothers)
# ---------------------------------------------------------------------------
# 1) Fingerprints
#    Two sales calls that "feel similar" should get similar z.
#    Then you can search: "find calls like this one" without comparing
#    raw audio.
#
# 2) Denoising
#    Feed a noisy version in, ask it to rebuild the clean version.
#    Like teaching it to ignore static on a phone line.
#
# 3) "That's weird"
#    If a new example will NOT rebuild well, it doesn't look like training
#    data. Useful as a simple alarm (odd audio, odd screenshot, odd log).
#
# Notice: in 1 and 3 you might throw the rebuild away and only keep z
# or the error score.
#
#
# ---------------------------------------------------------------------------
# How it trains (same homework energy as MiniGPT, different question)
# ---------------------------------------------------------------------------
# MiniGPT homework: "what word comes next?"
# Autoencoder homework: "can you redraw what I just showed you?"
#
# Show x → get x̂ → measure "how far is x̂ from x?" → twist knobs.
# Repeat. Encoder gets better at describing. Decoder gets better at drawing.
#
#
# ---------------------------------------------------------------------------
# NOW THE VAE — same sketch artist, but they can also invent faces
# ---------------------------------------------------------------------------
# Plain autoencoder: one input → one exact z → one rebuild.
#   It's a photocopier with a tiny memory. Great at rebuilding. Awkward at
#   inventing a NEW face it never saw.
#
# Why awkward? Its z's are like secret locker numbers scattered all over
# a huge building. If you pick a random locker, it's empty. Decode garbage.
#
# VAE changes the rule:
#   Don't give one locker number.
#   Give a NEIGHBORHOOD: "around locker 12, plus or minus a bit."
#
# Then:
#   - at train time we pick a random locker in that neighborhood and decode
#   - later we can wander the hallway of neighborhoods and decode NEW faces
#
# Two extra names (only because papers use them):
#   μ  (mu)    = the middle of the neighborhood
#   σ  (sigma) = how wide the neighborhood is
#   sample z   = "stand near μ, take a small random step of size σ"
#
# Extra homework for a VAE (besides "rebuild well"):
#   keep those neighborhoods in the SAME part of town, overlapping a bit,
#   not one house in Tokyo and one in Lima. Then a random walk still
#   lands in a real neighborhood → decoder draws something plausible.
#
#
# Example, same 4 pixels:
#   AE:   x → z = [0.8, -0.2]              → rebuild
#   VAE:  x → μ = [0.8, -0.2],  σ = [0.1, 0.1]
#         pick z = [0.83, -0.17] this time  → rebuild
#         next time maybe [0.74, -0.25]     → slightly different rebuild
#
# Mix two neighborhoods (smile + glasses) → decoder can draw "smiling
# person with glasses" even if that exact photo wasn't in the dataset.
# That's the generative trick.
#
#
# ---------------------------------------------------------------------------
# One table to keep them straight
# ---------------------------------------------------------------------------
#   What you want                         Use
#   ----------------------------------    ------------------------
#   Finish a sentence                     GPT
#   Compress / fingerprint / denoise      Autoencoder
#   Compress AND invent new examples      VAE
#
#
# ---------------------------------------------------------------------------

In [3]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# MNIST: images are [0,1], shape [1, 28, 28]
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

Using device: cpu
Train samples: 60000, Test samples: 10000


In [4]:
# Autoencoder from Scratch
# We implement a simple MLP autoencoder with a bottleneck of size 32.
class Autoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()   # outputs in [0, 1] to match MNIST pixels
        )

    def forward(self, x):
        # x: [batch, 1, 28, 28] -> flatten to [batch, 784]
        z = self.encoder(x.view(x.size(0), -1))
        x_recon = self.decoder(z)
        return x_recon